# Titanic - Train on Colab GPU

Mounts Google Drive and trains the `TabularResNet` using the code + data you uploaded.

**Upload this folder to your Drive at `MyDrive/IAI_TITANIC/`:**
```
MyDrive/IAI_TITANIC/
+-- src/                 (preprocessing.py, model.py, __init__.py)
+-- train.py
+-- prepare_data.py
+-- requirements.txt
+-- data/train.csv       (already-fetched data)
+-- artifacts/           (OPTIONAL: splits.npz + preprocessor.joblib from local `python prepare_data.py`)
```

Set the GPU runtime first: **Runtime -> Change runtime type -> GPU**.

## 1. Mount Drive + enter project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/IAI_TITANIC'
%cd $PROJECT
!pip install -q torch scikit-learn pandas numpy joblib optuna livelossplot wandb

## 2a. Train from locally-preprocessed cache (preferred)

Run `python prepare_data.py` on your machine first, then upload `artifacts/splits.npz` and `artifacts/preprocessor.joblib`. No Kaggle creds, no preprocessing in Colab.

In [ ]:
from train import train

res = train(
    splits_path='artifacts/splits.npz',
    preprocessor_path='artifacts/preprocessor.joblib',
    out_dir='artifacts',
    epochs=200, patience=20, device='cuda',
)
res['best_metrics']

## 2b. OR preprocess + train in Colab from raw CSV

Use this instead of 2a if you only uploaded `data/train.csv` (no cached arrays).

In [ ]:
from train import train

res = train(
    data_path='data/train.csv',
    out_dir='artifacts',
    epochs=200, patience=20, device='cuda',
)
res['best_metrics']

## 2c. Hyperparameters tuning

In [ ]:
import optuna
from train import train

def objective(trial):
    # הגדרת מרחב החיפוש להיפר-פרמטרים
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.1, 0.3)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    
    # הרצת האימון עם הפרמטרים של ה-Trial הנוכחי
    res = train(
        splits_path='artifacts/splits.npz',
        preprocessor_path='artifacts/preprocessor.joblib',
        out_dir=f'artifacts/trial_{trial.number}',
        epochs=100, # עדיף פחות אופוקים לטובת חיפוש מהיר
        patience=10, 
        device='cuda',
        lr=lr, # בהנחה ש-train.py יודע לקבל אותם
        dropout=dropout,
        batch_size=batch_size,
        verbose=False # נכבה הדפסות כדי לא להציף את המסך
    )
    
    # אנחנו רוצים למקסם את ה-Validation F1 או Accuracy
    return res['best_metrics']['val_accuracy']

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Best trial:")
print(study.best_trial.value)
print(study.best_trial.params)

## 3. Check saved artifacts

Everything is written back to `artifacts/` on your Drive (survives runtime shutdown).

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import numpy as np

# 1. הצגת הפרמטרים שנבחרו
with open('artifacts/metadata.json', 'r') as f:
    metadata = json.load(f)
print("Training Output:", json.dumps(metadata, indent=2))

# 2. אם הרצנו Optuna, נציג את חשיבות ההיפר-פרמטרים
try:
    from optuna.visualization import plot_param_importances
    fig = plot_param_importances(study)
    fig.show()
except NameError:
    pass

# הוספת קוד לטעינת ה-Validation Set והפקת Confusion Matrix + ROC Curve
# (מצריך לטעון את המודל מ-artifacts/best_model.pt ולהעביר עליו את נתוני ה-Val)